# Topic Identificator (Modeling)

#### By: Eric Michel
September 18, 2024

### Tasks performed in this project:

1. Read the .csv file using Pandas. Take a look at the top few records.
2. Normalize casings for the review text and extract the text into a list for easier manipulation.
3. Tokenize the reviews using NLTKs word_tokenize function.
4. Perform parts-of-speech tagging on each sentence using the NLTK POS tagger.
5. For the topic model, we should want to include only nouns.
- Find out all the POS tags that correspond to nouns.
- Limit the data to only terms with these tags.
6. Lemmatize.
- Different forms of the terms need to be treated as one.
- No need to provide POS tag to lemmatizer for now.
7. Remove stopwords and punctuation (if there are any).
8. Create a topic model using LDA on the cleaned-up data with 12 topics.
- Print out the top terms for each topic.
- What is the coherence of the model with the c_v metric?
9. Analyze the topics through the business lens.
- Determine which of the topics can be combined.
10. Create topic model using LDA with what you think is the optimal number of topics
    - What is the coherence of the model?
11. The business should be able to interpret the topics.
    - Name each of the identified topics.              
    - Create a table with the topic name and the top 10 terms in each to present to the business. 

## About the dataset

The dataset (K8 Reviews v0.2.csv) contains product reviews from Amazon


More real data can be obtained using:

https://github.com/scrapehero-code/amazon-review-scraper

https://github.com/aesuli/amadown2py

References:

https://github.com/eaglewarrior/NLP_CODES_PROJECTS/blob/master/4%20Topic%20Modeling.ipynb

## Intro
I perform topic modeling using Latent Dirichlet Allocation (LDA) based on the tasks outlined above.

In [1]:
## Install dependencies

# !pip install pandas nltk gensim scikit-learn


In [2]:
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

import string
import re

In [3]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/oysterable/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/oysterable/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/oysterable/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/oysterable/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# 1. Read the CSV file and inspect the data
df = pd.read_csv('./Datasets/K8ReviewsV02.csv')
print(df.head(10))  # Inspect the first few records

   sentiment                                             review
0          1             Good but need updates and improvements
1          0  Worst mobile i have bought ever, Battery is dr...
2          1  when I will get my 10% cash back.... its alrea...
3          1                                               Good
4          0  The worst phone everThey have changed the last...
5          0  Only I'm telling don't buyI'm totally disappoi...
6          1  Phone is awesome. But while charging, it heats...
7          0                    The battery level has worn down
8          0  It's over hitting problems...and phone hanging...
9          0  A lot of glitches dont buy this thing better g...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14675 entries, 0 to 14674
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  14675 non-null  int64 
 1   review     14675 non-null  object
dtypes: int64(1), object(1)
memory usage: 229.4+ KB


In [6]:
df.sentiment.value_counts()

sentiment
0    7712
1    6963
Name: count, dtype: int64

In [7]:
# Get the reviews as a list of strings and eliminate any null values
reviews = df['review'].dropna().astype(str).tolist()
reviews

['Good but need updates and improvements',
 "Worst mobile i have bought ever, Battery is draining like hell, backup is only 6 to 7 hours with internet uses, even if I put mobile idle its getting discharged.This is biggest lie from Amazon & Lenove which is not at all expected, they are making full by saying that battery is 4000MAH & booster charger is fake, it takes at least 4 to 5 hours to be fully charged.Don't know how Lenovo will survive by making full of us.Please don;t go for this else you will regret like me.",
 'when I will get my 10% cash back.... its already 15 January..',
 'Good',
 'The worst phone everThey have changed the last phone but the problem is still same and the amazon is not returning the phone .Highly disappointing of amazon',
 "Only I'm telling don't buyI'm totally disappointedPoor batteryPoor cameraWaste of money",
 'Phone is awesome. But while charging, it heats up allot..Really a genuine reason to hate Lenovo k8 note',
 'The battery level has worn down',
 "It'

In [8]:
# # 2. Normalize casings
# reviews = [review.lower() for review in reviews]
# reviews

In [9]:
#Preprocess the text data
def preprocess_text(document):
    
    text_out = []
    
    for text in document:

        # 2. Normalize casings
        text = text.lower()  # Convert to lowercase
        
        text = re.sub(r'\W', ' ', text)  # Remove punctuation and special characters
        text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
        # text = re.sub(r'\d', '', text)  # Remove numbers
        
        # 7. Remove stopwords and punctuation
        stop_words = set(stopwords.words('english'))
        punctuation = set(string.punctuation)

        words = text.split()        
        text = ' '.join([word for word in words if word not in stop_words and word not in punctuation])

        text_out.append(text)
    
    return text_out


reviews = preprocess_text(reviews)
reviews

['good need updates improvements',
 'worst mobile bought ever battery draining like hell backup 6 7 hours internet uses even put mobile idle getting discharged biggest lie amazon lenove expected making full saying battery 4000mah booster charger fake takes least 4 5 hours fully charged know lenovo survive making full us please go else regret like',
 'get 10 cash back already 15 january',
 'good',
 'worst phone everthey changed last phone problem still amazon returning phone highly disappointing amazon',
 'telling buyi totally disappointedpoor batterypoor camerawaste money',
 'phone awesome charging heats allot really genuine reason hate lenovo k8 note',
 'battery level worn',
 'hitting problems phone hanging problems lenovo k 8 note service station ahmedabad one years warranty change phone lenovo',
 'lot glitches dont buy thing better go options',
 'wrost',
 'good phone charger working damage within 2 months',
 'purchase item much heating battery life poor',
 'faced battery problem mot

In [10]:

# 3. Tokenize the reviews
tokenized_reviews = [word_tokenize(review) for review in reviews]
tokenized_reviews


[['good', 'need', 'updates', 'improvements'],
 ['worst',
  'mobile',
  'bought',
  'ever',
  'battery',
  'draining',
  'like',
  'hell',
  'backup',
  '6',
  '7',
  'hours',
  'internet',
  'uses',
  'even',
  'put',
  'mobile',
  'idle',
  'getting',
  'discharged',
  'biggest',
  'lie',
  'amazon',
  'lenove',
  'expected',
  'making',
  'full',
  'saying',
  'battery',
  '4000mah',
  'booster',
  'charger',
  'fake',
  'takes',
  'least',
  '4',
  '5',
  'hours',
  'fully',
  'charged',
  'know',
  'lenovo',
  'survive',
  'making',
  'full',
  'us',
  'please',
  'go',
  'else',
  'regret',
  'like'],
 ['get', '10', 'cash', 'back', 'already', '15', 'january'],
 ['good'],
 ['worst',
  'phone',
  'everthey',
  'changed',
  'last',
  'phone',
  'problem',
  'still',
  'amazon',
  'returning',
  'phone',
  'highly',
  'disappointing',
  'amazon'],
 ['telling',
  'buyi',
  'totally',
  'disappointedpoor',
  'batterypoor',
  'camerawaste',
  'money'],
 ['phone',
  'awesome',
  'charging

In [11]:

# 4. Perform POS tagging
tagged_reviews = [pos_tag(tokens) for tokens in tokenized_reviews]
tagged_reviews


[[('good', 'JJ'), ('need', 'NN'), ('updates', 'NNS'), ('improvements', 'NNS')],
 [('worst', 'RB'),
  ('mobile', 'NN'),
  ('bought', 'VBD'),
  ('ever', 'RB'),
  ('battery', 'RB'),
  ('draining', 'VBG'),
  ('like', 'IN'),
  ('hell', 'NN'),
  ('backup', 'NN'),
  ('6', 'CD'),
  ('7', 'CD'),
  ('hours', 'NNS'),
  ('internet', 'JJ'),
  ('uses', 'NNS'),
  ('even', 'RB'),
  ('put', 'VBP'),
  ('mobile', 'JJ'),
  ('idle', 'JJ'),
  ('getting', 'VBG'),
  ('discharged', 'JJ'),
  ('biggest', 'JJS'),
  ('lie', 'NN'),
  ('amazon', 'NN'),
  ('lenove', 'NN'),
  ('expected', 'VBD'),
  ('making', 'VBG'),
  ('full', 'JJ'),
  ('saying', 'VBG'),
  ('battery', 'NN'),
  ('4000mah', 'CD'),
  ('booster', 'NN'),
  ('charger', 'NN'),
  ('fake', 'VBP'),
  ('takes', 'VBZ'),
  ('least', 'JJS'),
  ('4', 'CD'),
  ('5', 'CD'),
  ('hours', 'NNS'),
  ('fully', 'RB'),
  ('charged', 'VBN'),
  ('know', 'VBP'),
  ('lenovo', 'JJ'),
  ('survive', 'JJ'),
  ('making', 'VBG'),
  ('full', 'JJ'),
  ('us', 'PRP'),
  ('please', 'VB'),

In [12]:

# 5. Extract only nouns
# NLTK POS tags for nouns include: NN, NNS, NNP, NNPS
nouns = [[word for word, pos in tagged if pos in ['NN', 'NNS', 'NNP', 'NNPS']] for tagged in tagged_reviews]
nouns

[['need', 'updates', 'improvements'],
 ['mobile',
  'hell',
  'backup',
  'hours',
  'uses',
  'lie',
  'amazon',
  'lenove',
  'battery',
  'booster',
  'charger',
  'hours',
  'regret'],
 ['cash'],
 [],
 ['phone', 'everthey', 'phone', 'problem', 'phone', 'amazon'],
 ['buyi', 'batterypoor', 'camerawaste', 'money'],
 ['phone', 'heats', 'reason', 'hate', 'lenovo', 'k8', 'note'],
 ['battery', 'level', 'worn'],
 ['problems',
  'phone',
  'problems',
  'note',
  'service',
  'station',
  'years',
  'change',
  'phone',
  'lenovo'],
 ['lot', 'glitches', 'thing', 'options'],
 ['wrost'],
 ['phone', 'charger', 'damage', 'months'],
 ['purchase', 'item', 'heating', 'battery', 'life'],
 ['battery', 'problem', 'motherboard', 'problem', 'months', 'life'],
 ['phone', 'slim', 'battry', 'backup', 'screen', 'love'],
 ['headset'],
 ['time'],
 ['product',
  'range',
  'specification',
  'comparison',
  'range',
  'phone',
  'amazon',
  'seal',
  'credit',
  'card',
  'rs',
  'deal',
  'amazon'],
 ['batte

In [13]:
# 6. Lemmatize the nouns
lemmatizer = WordNetLemmatizer()
cleaned_reviews = [[lemmatizer.lemmatize(noun) for noun in review] for review in nouns]
cleaned_reviews

[['need', 'update', 'improvement'],
 ['mobile',
  'hell',
  'backup',
  'hour',
  'us',
  'lie',
  'amazon',
  'lenove',
  'battery',
  'booster',
  'charger',
  'hour',
  'regret'],
 ['cash'],
 [],
 ['phone', 'everthey', 'phone', 'problem', 'phone', 'amazon'],
 ['buyi', 'batterypoor', 'camerawaste', 'money'],
 ['phone', 'heat', 'reason', 'hate', 'lenovo', 'k8', 'note'],
 ['battery', 'level', 'worn'],
 ['problem',
  'phone',
  'problem',
  'note',
  'service',
  'station',
  'year',
  'change',
  'phone',
  'lenovo'],
 ['lot', 'glitch', 'thing', 'option'],
 ['wrost'],
 ['phone', 'charger', 'damage', 'month'],
 ['purchase', 'item', 'heating', 'battery', 'life'],
 ['battery', 'problem', 'motherboard', 'problem', 'month', 'life'],
 ['phone', 'slim', 'battry', 'backup', 'screen', 'love'],
 ['headset'],
 ['time'],
 ['product',
  'range',
  'specification',
  'comparison',
  'range',
  'phone',
  'amazon',
  'seal',
  'credit',
  'card',
  'r',
  'deal',
  'amazon'],
 ['battery', 'solution',

In [14]:
# # 6. Lemmatize the nouns
# lemmatizer = WordNetLemmatizer()
# lemmatized_reviews = [[lemmatizer.lemmatize(noun) for noun in review] for review in nouns]
# lemmatized_reviews

In [15]:
# # 7. Remove stopwords and punctuation
# stop_words = set(stopwords.words('english'))
# punctuation = set(string.punctuation)

# cleaned_reviews = [[word for word in review if word not in stop_words and word not in punctuation] for review in lemmatized_reviews]
# cleaned_reviews

8. Create a topic model using LDA on the cleaned-up data with 12 topics.
- Print out the top terms for each topic.
- What is the coherence of the model with the c_v metric?

In [16]:

# Now the cleaned_reviews can be used for the LDA topic modeling (steps 8 to 11)
from gensim import corpora
from gensim.models.ldamodel import LdaModel
from gensim.models import CoherenceModel


In [17]:

# Create a dictionary and a bag-of-words corpus for the LDA model with 12 topics
dictionary = corpora.Dictionary(cleaned_reviews)
dictionary

In [18]:
#BOW
corpus = [dictionary.doc2bow(review) for review in cleaned_reviews]
corpus

[[(0, 1), (1, 1), (2, 1)],
 [(3, 1),
  (4, 1),
  (5, 1),
  (6, 1),
  (7, 1),
  (8, 1),
  (9, 2),
  (10, 1),
  (11, 1),
  (12, 1),
  (13, 1),
  (14, 1)],
 [(15, 1)],
 [],
 [(3, 1), (16, 1), (17, 3), (18, 1)],
 [(19, 1), (20, 1), (21, 1), (22, 1)],
 [(17, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1)],
 [(5, 1), (29, 1), (30, 1)],
 [(17, 2), (18, 2), (26, 1), (27, 1), (31, 1), (32, 1), (33, 1), (34, 1)],
 [(35, 1), (36, 1), (37, 1), (38, 1)],
 [(39, 1)],
 [(7, 1), (17, 1), (40, 1), (41, 1)],
 [(5, 1), (42, 1), (43, 1), (44, 1), (45, 1)],
 [(5, 1), (18, 2), (41, 1), (44, 1), (46, 1)],
 [(4, 1), (17, 1), (47, 1), (48, 1), (49, 1), (50, 1)],
 [(51, 1)],
 [(52, 1)],
 [(3, 2),
  (17, 1),
  (53, 1),
  (54, 1),
  (55, 1),
  (56, 1),
  (57, 1),
  (58, 1),
  (59, 2),
  (60, 1),
  (61, 1)],
 [(5, 1), (44, 1), (62, 1)],
 [(63, 1)],
 [],
 [(17, 1), (18, 1), (64, 1), (65, 1), (66, 1)],
 [(5, 1), (67, 1), (68, 1), (69, 1)],
 [(57, 1)],
 [(5, 1), (17, 1), (26, 1), (57, 2), (67, 1)],
 [(37, 2

In [19]:
# Train the LDA model
lda_model = LdaModel(corpus, num_topics=12, id2word=dictionary, passes=10, random_state=100)

# Coherence score with c_v metric
coherence_model_lda = CoherenceModel(model=lda_model, texts=cleaned_reviews, dictionary=dictionary, coherence='c_v')

coherence_lda = coherence_model_lda.get_coherence()
print(f"Coherence Score: {coherence_lda}")

Coherence Score: 0.5058918235771355


In [20]:
# Print the top terms for each topic
topics = lda_model.print_topics(num_words=10)
topics

[(0,
  '0.162*"mobile" + 0.047*"sim" + 0.034*"stock" + 0.033*"android" + 0.031*"jio" + 0.031*"software" + 0.026*"awesome" + 0.024*"super" + 0.023*"fast" + 0.023*"bit"'),
 (1,
  '0.077*"waste" + 0.057*"money" + 0.042*"dolby" + 0.034*"lenovo" + 0.033*"please" + 0.033*"experience" + 0.033*"thanks" + 0.021*"dont" + 0.019*"smartphone" + 0.016*"request"'),
 (2,
  '0.363*"phone" + 0.046*"time" + 0.042*"charge" + 0.036*"charger" + 0.028*"heat" + 0.025*"use" + 0.024*"month" + 0.024*"issue" + 0.020*"turbo" + 0.020*"hour"'),
 (3,
  '0.176*"phone" + 0.162*"price" + 0.070*"money" + 0.065*"range" + 0.063*"performance" + 0.041*"superb" + 0.040*"value" + 0.025*"budget" + 0.024*"buy" + 0.017*"earphone"'),
 (4,
  '0.269*"product" + 0.060*"service" + 0.056*"amazon" + 0.036*"return" + 0.030*"delivery" + 0.027*"customer" + 0.019*"day" + 0.019*"replacement" + 0.018*"time" + 0.017*"center"'),
 (5,
  '0.231*"camera" + 0.094*"quality" + 0.071*"phone" + 0.044*"battery" + 0.021*"performance" + 0.021*"display" + 

In [21]:
initial_topics = {}
for idx, topic in topics:
    # print(f"Topic {idx}: {topic}")
    initial_topics[idx]=[topic]
initial_topics


{0: ['0.162*"mobile" + 0.047*"sim" + 0.034*"stock" + 0.033*"android" + 0.031*"jio" + 0.031*"software" + 0.026*"awesome" + 0.024*"super" + 0.023*"fast" + 0.023*"bit"'],
 1: ['0.077*"waste" + 0.057*"money" + 0.042*"dolby" + 0.034*"lenovo" + 0.033*"please" + 0.033*"experience" + 0.033*"thanks" + 0.021*"dont" + 0.019*"smartphone" + 0.016*"request"'],
 2: ['0.363*"phone" + 0.046*"time" + 0.042*"charge" + 0.036*"charger" + 0.028*"heat" + 0.025*"use" + 0.024*"month" + 0.024*"issue" + 0.020*"turbo" + 0.020*"hour"'],
 3: ['0.176*"phone" + 0.162*"price" + 0.070*"money" + 0.065*"range" + 0.063*"performance" + 0.041*"superb" + 0.040*"value" + 0.025*"budget" + 0.024*"buy" + 0.017*"earphone"'],
 4: ['0.269*"product" + 0.060*"service" + 0.056*"amazon" + 0.036*"return" + 0.030*"delivery" + 0.027*"customer" + 0.019*"day" + 0.019*"replacement" + 0.018*"time" + 0.017*"center"'],
 5: ['0.231*"camera" + 0.094*"quality" + 0.071*"phone" + 0.044*"battery" + 0.021*"performance" + 0.021*"display" + 0.019*"proce

In [22]:

# 9. Analyze and combine topics based on business understanding
# Combine topics into broader business-related themes
combined_topics = {
    'Product Quality': initial_topics[2]+initial_topics[5]+initial_topics[9],
    'Product Features': initial_topics[0]+initial_topics[6]+initial_topics[8]+initial_topics[10],
    'Reviews and Feedback': initial_topics[1]+initial_topics[3]+initial_topics[7]+initial_topics[11], 
    'Customer Service and Returns': initial_topics[4]
}

# Now, print out the newly combined topics with their top terms for business understanding
for topic_name, terms in combined_topics.items():
    print(f"Topic: {topic_name}")
    print(f"Top terms: {', '.join(terms[:10])}")  # Show only top 10 terms
    print("\n")

Topic: Product Quality
Top terms: 0.363*"phone" + 0.046*"time" + 0.042*"charge" + 0.036*"charger" + 0.028*"heat" + 0.025*"use" + 0.024*"month" + 0.024*"issue" + 0.020*"turbo" + 0.020*"hour", 0.231*"camera" + 0.094*"quality" + 0.071*"phone" + 0.044*"battery" + 0.021*"performance" + 0.021*"display" + 0.019*"processor" + 0.016*"mode" + 0.015*"sound" + 0.015*"speed", 0.076*"h" + 0.071*"good" + 0.051*"headphone" + 0.041*"condition" + 0.037*"worth" + 0.034*"plz" + 0.031*"class" + 0.027*"look" + 0.024*"n" + 0.022*"facility"


Topic: Product Features
Top terms: 0.162*"mobile" + 0.047*"sim" + 0.034*"stock" + 0.033*"android" + 0.031*"jio" + 0.031*"software" + 0.026*"awesome" + 0.024*"super" + 0.023*"fast" + 0.023*"bit", 0.094*"screen" + 0.049*"hai" + 0.041*"glass" + 0.025*"cast" + 0.022*"gorilla" + 0.020*"cost" + 0.019*"ho" + 0.015*"hand" + 0.015*"tv" + 0.015*"display", 0.139*"note" + 0.088*"lenovo" + 0.076*"k8" + 0.062*"call" + 0.037*"option" + 0.022*"system" + 0.016*"music" + 0.016*"model" + 0

In [23]:

# 10. Recreate topic model with the optimal number of topics

optimal_num_topics = len(combined_topics)
optimal_lda_model = LdaModel(corpus, num_topics=optimal_num_topics, id2word=dictionary, passes=10, random_state=100)


# Print top terms for each topic
optimal_topics = optimal_lda_model.print_topics(num_words=10)
for idx, topic in optimal_topics:
    print(f"Topic {idx}: {topic}")

Topic 0: 0.110*"camera" + 0.065*"battery" + 0.059*"phone" + 0.045*"quality" + 0.030*"performance" + 0.025*"mobile" + 0.019*"feature" + 0.019*"backup" + 0.012*"mode" + 0.010*"display"
Topic 1: 0.064*"note" + 0.036*"lenovo" + 0.035*"k8" + 0.028*"phone" + 0.015*"call" + 0.014*"hai" + 0.011*"h" + 0.008*"processor" + 0.008*"feature" + 0.008*"option"
Topic 2: 0.128*"phone" + 0.047*"problem" + 0.041*"issue" + 0.036*"battery" + 0.027*"time" + 0.023*"day" + 0.017*"network" + 0.015*"month" + 0.015*"charge" + 0.014*"hour"
Topic 3: 0.186*"product" + 0.077*"price" + 0.052*"money" + 0.051*"phone" + 0.031*"range" + 0.026*"waste" + 0.021*"delivery" + 0.019*"superb" + 0.019*"everything" + 0.019*"value"


In [24]:

# Coherence score for the new model
coherence_model_optimal = CoherenceModel(model=optimal_lda_model, texts=cleaned_reviews, dictionary=dictionary, coherence='c_v')
coherence_optimal = coherence_model_optimal.get_coherence()
print(f"Optimal Model Coherence Score: {coherence_optimal}")


Optimal Model Coherence Score: 0.5882628824654401


In [25]:

# 11. Assigning business-friendly names to topics (domain knowledge required)
topic_names = {
    0: "Product Quality",
    1: "Product Features",
    2: "Reviews and Feedback",
    3: "Customer Service and Returns"
}

# Create a table with topic names and top 10 terms
topic_table = []
for idx, topic in optimal_topics:
    topic_terms = [term.split('*')[1].strip().replace('"', '') for term in topic.split('+')]
    topic_table.append((topic_names.get(idx, f"Topic {idx}"), topic_terms))

# Display topic names and terms
for topic_name, terms in topic_table:
    print(f"Topic: {topic_name}")
    print(f"Top terms: {', '.join(terms)}")
    print("\n")

Topic: Product Quality
Top terms: camera, battery, phone, quality, performance, mobile, feature, backup, mode, display


Topic: Product Features
Top terms: note, lenovo, k8, phone, call, hai, h, processor, feature, option


Topic: Reviews and Feedback
Top terms: phone, problem, issue, battery, time, day, network, month, charge, hour


Topic: Customer Service and Returns
Top terms: product, price, money, phone, range, waste, delivery, superb, everything, value


